# 📍 Phase 2 — Géo-Enrichissement (OpenStreetMap)
## Projet Capstone : Prédiction des Prix Immobiliers en Mauritanie
### Master 1 Machine Learning — SupNum

---

> **Objectif :** Transformer les adresses textuelles en coordonnées GPS et enrichir le dataset avec des features géographiques (distances, POIs) via OpenStreetMap.

> **Input :** `raw_data.csv` (sortie de la Phase 1)  
> **Output :** `enriched_data.csv` (dataset enrichi avec features géographiques)

---

## 1. 📦 Import des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

# Géo-spatial
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import requests
import json

# Visualisation
import folium
from folium.plugins import HeatMap
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

print('✅ Bibliothèques importées')

## 2. 📂 Chargement des données

On charge le dataset brut issu de la Phase 1 (scraping). Si vous n'avez pas encore de `raw_data.csv`, on utilise le dataset Kaggle comme base.

In [ ]:
# Adapter le chemin selon votre structure
# Si raw_data.csv existe (Phase 1 terminée) :
# df = pd.read_csv('data/raw/raw_data.csv')

# Sinon, utiliser le dataset Kaggle :
df = pd.read_csv('kaggle_train.csv')

print(f'Dataset chargé : {df.shape[0]} annonces × {df.shape[1]} colonnes')
print(f'Colonnes : {df.columns.tolist()}')
print(f'\nQuartiers uniques : {df["quartier"].unique().tolist()}')
df.head()

## 3. 🗺️ Géocodage — Coordonnées GPS des Quartiers

### 3.1 Coordonnées de référence

Les quartiers de Nouakchott sont bien définis. On utilise des coordonnées de référence fiables plutôt que le géocodage automatique (qui peut être imprécis pour la Mauritanie).

In [ ]:
# ══════════════════════════════════════════════════════════════
# Coordonnées de référence — Quartiers de Nouakchott
# Sources : OpenStreetMap + vérification manuelle
# ══════════════════════════════════════════════════════════════
QUARTIERS_GPS = {
    'Tevragh Zeina': {'lat': 18.1036, 'lon': -15.9785, 'desc': 'Quartier huppé, ambassades'},
    'Ksar':          {'lat': 18.0866, 'lon': -15.9750, 'desc': 'Centre historique'},
    'Arafat':        {'lat': 18.0550, 'lon': -15.9610, 'desc': 'Populaire, dense'},
    'Dar Naim':      {'lat': 18.1200, 'lon': -15.9450, 'desc': 'Résidentiel, en expansion'},
    'Toujounine':    {'lat': 18.0680, 'lon': -15.9350, 'desc': 'Périphérie, récent'},
    'Sebkha':        {'lat': 18.0730, 'lon': -15.9870, 'desc': 'Commercial, marchés'},
    'El Mina':       {'lat': 18.0580, 'lon': -16.0050, 'desc': 'Bord de mer, populaire'},
    'Riyadh':        {'lat': 18.0850, 'lon': -15.9550, 'desc': 'Résidentiel moyen'},
    'Teyarett':      {'lat': 18.0950, 'lon': -15.9700, 'desc': 'Centre, mixte'},
}

# Points de référence pour le calcul de distances
POINTS_REF = {
    'centre_ville': (18.0866, -15.9750),      # Ksar (centre historique)
    'aeroport':     (18.0983, -15.9475),       # Aéroport International Oumtounsy
    'plage':        (18.0580, -16.0150),       # Plage de Nouakchott
    'grand_marche': (18.0860, -15.9780),       # Grand Marché (Capitale)
    'port':         (18.0350, -16.0250),       # Port de l'Amitié
}

print('Coordonnées de référence :')
for q, info in QUARTIERS_GPS.items():
    print(f'  {q:<18} ({info["lat"]:.4f}, {info["lon"]:.4f}) — {info["desc"]}')

### 3.2 Géocodage avec Nominatim (vérification)

On utilise l'API Nominatim d'OpenStreetMap pour vérifier et compléter les coordonnées. Cela est utile si des quartiers supplémentaires apparaissent dans les données.

In [ ]:
# ── Géocodeur Nominatim ────────────────────────────────────────────────────────
geolocator = Nominatim(user_agent="mauritania_housing_ml_supnum_2026")

def geocode_quartier(quartier, ville="Nouakchott, Mauritanie"):
    """Géocode un quartier via Nominatim avec fallback sur les coordonnées de référence."""
    # D'abord vérifier nos coordonnées de référence
    for q_ref, info in QUARTIERS_GPS.items():
        if quartier.lower().strip() == q_ref.lower().strip():
            return info['lat'], info['lon'], 'reference'
    
    # Sinon, essayer Nominatim
    try:
        query = f"{quartier}, {ville}"
        location = geolocator.geocode(query, timeout=10)
        time.sleep(1)  # Respecter le rate-limit de Nominatim
        if location:
            return location.latitude, location.longitude, 'nominatim'
    except Exception as e:
        print(f"  ⚠️ Erreur géocodage {quartier}: {e}")
    
    # Fallback : coordonnées du centre de Nouakchott
    return 18.0866, -15.9750, 'fallback'

# ── Appliquer le géocodage ────────────────────────────────────────────────────
print('Géocodage des quartiers...')
geocoded = {}
for quartier in df['quartier'].unique():
    lat, lon, source = geocode_quartier(quartier)
    geocoded[quartier] = {'lat': lat, 'lon': lon, 'source': source}
    print(f'  {quartier:<18} → ({lat:.4f}, {lon:.4f}) [{source}]')

# Ajouter les coordonnées au dataset
df['latitude'] = df['quartier'].map(lambda q: geocoded.get(q, {}).get('lat', 18.0866))
df['longitude'] = df['quartier'].map(lambda q: geocoded.get(q, {}).get('lon', -15.9750))

print(f'\n✅ Géocodage terminé : {len(geocoded)} quartiers')

## 4. 📏 Calcul des Distances

On calcule la distance géodésique (en km) entre chaque bien et les points de référence importants de Nouakchott.

In [ ]:
# ══════════════════════════════════════════════════════════════
# FEATURES DE DISTANCE
# ══════════════════════════════════════════════════════════════
def calc_distance_km(lat, lon, ref_point):
    """Calcule la distance en km entre un point et un point de référence."""
    try:
        return geodesic((lat, lon), ref_point).kilometers
    except:
        return np.nan

# Calculer les distances pour chaque annonce
for ref_name, ref_coords in POINTS_REF.items():
    col_name = f'dist_{ref_name}_km'
    df[col_name] = df.apply(
        lambda row: calc_distance_km(row['latitude'], row['longitude'], ref_coords), axis=1
    )
    print(f'{col_name:<25} — min: {df[col_name].min():.2f} km, max: {df[col_name].max():.2f} km, mean: {df[col_name].mean():.2f} km')

print('\n✅ Features de distance calculées')

## 5. 🏪 Enrichissement via Overpass API (POIs OpenStreetMap)

L'API Overpass permet de récupérer les Points d'Intérêt (POI) autour d'un point GPS. On compte le nombre d'écoles, mosquées, commerces, hôpitaux dans un rayon donné.

In [ ]:
# ══════════════════════════════════════════════════════════════
# OVERPASS API — Récupération des POIs
# ══════════════════════════════════════════════════════════════
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

def count_pois_around(lat, lon, radius_m=1000, poi_type="amenity", poi_value=None):
    """
    Compte les POIs d'un type donné dans un rayon autour d'un point.
    
    Args:
        lat, lon: coordonnées GPS
        radius_m: rayon en mètres (default 1000m = 1km)
        poi_type: clé OSM (amenity, shop, building, etc.)
        poi_value: valeur OSM (school, mosque, hospital, etc.)
    
    Returns:
        int: nombre de POIs trouvés
    """
    if poi_value:
        query = f'''
        [out:json][timeout:25];
        (
          node["{poi_type}"="{poi_value}"](around:{radius_m},{lat},{lon});
          way["{poi_type}"="{poi_value}"](around:{radius_m},{lat},{lon});
        );
        out count;
        '''
    else:
        query = f'''
        [out:json][timeout:25];
        (
          node["{poi_type}"](around:{radius_m},{lat},{lon});
          way["{poi_type}"](around:{radius_m},{lat},{lon});
        );
        out count;
        '''
    
    try:
        response = requests.get(OVERPASS_URL, params={'data': query}, timeout=30)
        data = response.json()
        return data['elements'][0]['tags']['total'] if data.get('elements') else 0
    except Exception as e:
        return 0

# ── Définir les types de POI à chercher ───────────────────────────────────────
POI_TYPES = {
    'nb_ecoles_1km':    ('amenity', 'school'),
    'nb_mosquees_1km':  ('amenity', 'place_of_worship'),
    'nb_commerces_1km': ('shop', None),          # Tous les commerces
    'nb_hopitaux_1km':  ('amenity', 'hospital'),
    'nb_restaurants_1km': ('amenity', 'restaurant'),
    'nb_banks_1km':     ('amenity', 'bank'),
}

print('Récupération des POIs via Overpass API...')
print('(Cela peut prendre quelques minutes — on requête par quartier, pas par annonce)')
print()

# On requête par quartier (pas par annonce) pour éviter de surcharger l'API
quartier_pois = {}

for quartier, coords in geocoded.items():
    lat, lon = coords['lat'], coords['lon']
    quartier_pois[quartier] = {}
    
    for poi_name, (poi_type, poi_value) in POI_TYPES.items():
        count = count_pois_around(lat, lon, radius_m=1000, poi_type=poi_type, poi_value=poi_value)
        quartier_pois[quartier][poi_name] = int(count)
        time.sleep(1)  # Respecter le rate-limit
    
    total = sum(quartier_pois[quartier].values())
    quartier_pois[quartier]['nb_total_pois_1km'] = total
    
    print(f'  {quartier:<18} → {quartier_pois[quartier]}')

# Mapper sur le dataset
for poi_name in list(POI_TYPES.keys()) + ['nb_total_pois_1km']:
    df[poi_name] = df['quartier'].map(lambda q: quartier_pois.get(q, {}).get(poi_name, 0))

print('\n✅ Enrichissement POI terminé')

## 6. 🗺️ Visualisation Cartographique

### 6.1 Carte interactive des annonces

In [ ]:
# ══════════════════════════════════════════════════════════════
# CARTE FOLIUM — Annonces immobilières à Nouakchott
# ══════════════════════════════════════════════════════════════

# Centre de la carte
center_lat = df['latitude'].mean()
center_lon = df['longitude'].mean()

# Créer la carte
m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles='OpenStreetMap')

# Couleurs par quartier
colors = {
    'tevragh zeina': 'red', 'ksar': 'blue', 'arafat': 'green',
    'teyarett': 'purple', 'dar naim': 'orange', 'toujounine': 'cadetblue',
    'sebkha': 'darkgreen', 'riyadh': 'pink', 'el mina': 'gray'
}

# Ajouter les marqueurs (échantillon pour la lisibilité)
sample = df.sample(min(200, len(df)), random_state=42)
for _, row in sample.iterrows():
    quartier = str(row['quartier']).lower().strip()
    color = colors.get(quartier, 'gray')
    prix_str = f"{row['prix']:,.0f} MRU" if pd.notna(row.get('prix')) else 'N/A'
    
    popup_html = f"""
    <b>{row.get('titre', 'Annonce')[:50]}</b><br>
    Quartier: {row['quartier']}<br>
    Prix: {prix_str}<br>
    Surface: {row.get('surface_m2', 'N/A')} m²
    """
    
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        color=color,
        fill=True,
        fillOpacity=0.7,
        popup=folium.Popup(popup_html, max_width=300)
    ).add_to(m)

# Ajouter les points de référence
for ref_name, ref_coords in POINTS_REF.items():
    folium.Marker(
        location=list(ref_coords),
        popup=ref_name.replace('_', ' ').title(),
        icon=folium.Icon(color='black', icon='star')
    ).add_to(m)

# Légende
legend_html = '<div style="position:fixed;bottom:30px;left:30px;z-index:1000;background:white;padding:10px;border-radius:5px;border:1px solid gray;">'
legend_html += '<b>Quartiers</b><br>'
for q, c in colors.items():
    legend_html += f'<span style="color:{c}">●</span> {q.title()}<br>'
legend_html += '</div>'
m.get_root().html.add_child(folium.Element(legend_html))

# Sauvegarder
m.save('carte_nouakchott_annonces.html')
print('✅ Carte sauvegardée : carte_nouakchott_annonces.html')
m

### 6.2 Heatmap des prix

In [ ]:
# ══════════════════════════════════════════════════════════════
# HEATMAP DES PRIX
# ══════════════════════════════════════════════════════════════
if 'prix' in df.columns:
    m_heat = folium.Map(location=[center_lat, center_lon], zoom_start=12)
    
    # Données pour la heatmap (lat, lon, poids=prix normalisé)
    heat_data = df[['latitude', 'longitude', 'prix']].dropna()
    heat_data['prix_norm'] = heat_data['prix'] / heat_data['prix'].max()
    
    HeatMap(
        heat_data[['latitude', 'longitude', 'prix_norm']].values.tolist(),
        radius=25, blur=15, max_zoom=13,
        gradient={0.2: 'blue', 0.4: 'lime', 0.6: 'yellow', 0.8: 'orange', 1: 'red'}
    ).add_to(m_heat)
    
    m_heat.save('heatmap_prix_nouakchott.html')
    print('✅ Heatmap sauvegardée : heatmap_prix_nouakchott.html')
    m_heat

### 6.3 Visualisations statistiques des features géographiques

In [ ]:
# ── Distances par quartier ─────────────────────────────────────────────────────
dist_cols = [c for c in df.columns if c.startswith('dist_')]

fig, axes = plt.subplots(1, len(dist_cols), figsize=(5*len(dist_cols), 5))
if len(dist_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, dist_cols):
    quartier_dist = df.groupby('quartier')[col].mean().sort_values()
    quartier_dist.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(col.replace('dist_', '').replace('_km', '').replace('_', ' ').title())
    ax.set_xlabel('Distance (km)')

plt.suptitle('Distance moyenne par quartier', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── POIs par quartier ─────────────────────────────────────────────────────────
poi_cols = [c for c in df.columns if c.startswith('nb_') and '1km' in c]

if poi_cols:
    poi_by_quartier = df.groupby('quartier')[poi_cols].mean()
    
    fig, ax = plt.subplots(figsize=(12, 6))
    poi_by_quartier.plot(kind='bar', ax=ax)
    ax.set_title('Nombre moyen de POIs par quartier (rayon 1km)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Nombre de POIs')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Corrélation des features géo avec le prix ─────────────────────────────────
if 'prix' in df.columns:
    geo_cols = dist_cols + poi_cols + ['latitude', 'longitude']
    geo_cols = [c for c in geo_cols if c in df.columns]
    
    correlations = df[geo_cols + ['prix']].corr()['prix'].drop('prix').sort_values()
    
    fig, ax = plt.subplots(figsize=(10, 6))
    correlations.plot(kind='barh', ax=ax, color=['coral' if v < 0 else 'steelblue' for v in correlations])
    ax.set_title('Corrélation des features géographiques avec le prix', fontsize=13, fontweight='bold')
    ax.axvline(x=0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.show()
    
    print('\nCorrélations avec le prix :')
    for col, corr in correlations.items():
        print(f'  {col:<25} {corr:+.4f}')

## 7. 📊 Résumé des features géographiques créées

| Feature | Description | Source |
|---|---|---|
| `latitude`, `longitude` | Coordonnées GPS du quartier | Coordonnées de référence |
| `dist_centre_ville_km` | Distance au centre (Ksar) | geodesic |
| `dist_aeroport_km` | Distance à l'aéroport Oumtounsy | geodesic |
| `dist_plage_km` | Distance à la plage | geodesic |
| `dist_grand_marche_km` | Distance au Grand Marché | geodesic |
| `dist_port_km` | Distance au Port de l'Amitié | geodesic |
| `nb_ecoles_1km` | Écoles dans un rayon de 1 km | Overpass API |
| `nb_mosquees_1km` | Mosquées dans un rayon de 1 km | Overpass API |
| `nb_commerces_1km` | Commerces proches | Overpass API |
| `nb_hopitaux_1km` | Hôpitaux et cliniques proches | Overpass API |
| `nb_restaurants_1km` | Restaurants proches | Overpass API |
| `nb_banks_1km` | Banques proches | Overpass API |
| `nb_total_pois_1km` | Total POI (proxy urbanisation) | Overpass API |

## 8. 💾 Sauvegarde du dataset enrichi

In [ ]:
# ══════════════════════════════════════════════════════════════
# SAUVEGARDE
# ══════════════════════════════════════════════════════════════
# Colonnes géographiques ajoutées
geo_features = ['latitude', 'longitude'] + dist_cols + poi_cols
if 'nb_total_pois_1km' in df.columns:
    geo_features.append('nb_total_pois_1km')

print(f'Features géographiques ajoutées : {len(geo_features)}')
print(geo_features)

# Sauvegarder
output_path = 'enriched_data.csv'
df.to_csv(output_path, index=False)
print(f'\n✅ Dataset enrichi sauvegardé : {output_path}')
print(f'   Shape : {df.shape}')
print(f'   Colonnes : {df.columns.tolist()}')

# Aperçu
print(f'\n=== Aperçu du dataset enrichi ===')
df[['quartier', 'latitude', 'longitude'] + dist_cols[:3]].drop_duplicates('quartier').sort_values('quartier')

## 9. 📋 Conclusion

### Ce qui a été fait dans cette phase :
1. **Géocodage** : Attribution de coordonnées GPS à chaque quartier via des coordonnées de référence fiables + vérification Nominatim
2. **Distances** : Calcul de 5 distances géodésiques (centre-ville, aéroport, plage, grand marché, port)
3. **POIs** : Récupération de 6 types de Points d'Intérêt via l'API Overpass d'OpenStreetMap
4. **Visualisation** : Carte interactive Folium + heatmap des prix + graphiques d'analyse
5. **Sauvegarde** : Dataset enrichi prêt pour la Phase 3 (EDA)

### Features créées : ~13 nouvelles variables géographiques

### Limites :
- Les coordonnées sont au niveau du quartier (pas de l'adresse exacte)
- Les données OSM pour Nouakchott sont incomplètes (peu de POIs référencés)
- Les distances sont à vol d'oiseau (pas en distance routière)

### Prochaine étape : Phase 3 — EDA Complète (`03_eda.ipynb`)